# CP Decomposition Benchmark — v3: Exact Python Ports (No MATLAB)

**Purpose.** Replaces the MATLAB bridge wrappers in v2 with exact Python/NumPy ports
of the actual MATLAB source code for TensorNOODL and M2PALS.

**What changed from v2:**
- MATLAB engine dependency removed entirely — runs anywhere including Google Colab.
- `run_tensor_noodl` and `run_m2pals` now call `tensor_methods_exact.py` functions
  which are line-by-line ports of the original `.m` files.
- Tensor generation for `noodl` mode now uses **Rademacher ±1** sparse factors
  (matching `tens_compare_NOODL.m`: `B = sign(randn.*binornd(1,alpha))`) instead
  of Gaussian — this is required by the algorithm's theoretical analysis.
- Mode unfoldings in M2PALS use Fortran (column-major) reshape order, matching MATLAB.

**Grid design (full):**
- Seeds: `[42, 26, 91, 7, 13, 100, 256, 314, 512, 1024]`
- Ranks: `[5, 10, 20, 50]`
- SNRs (dB): `[inf, 30, 20, 10]`
- Dims: `(50, 50, 50)` default; `(100, 100, 100)` for scalability sweep
- Sparsity (fraction nonzero): `[0.05, 0.10, 0.20]`


## 1. Setup

In [ ]:
import numpy as np
import scipy.io as sio
from scipy.optimize import linear_sum_assignment

import tensorly as tl
from tensorly.decomposition import parafac
from tensorly.cp_tensor import cp_to_tensor

import os, time, json, sys
from dataclasses import dataclass, asdict
from typing import Optional, List, Literal

import pandas as pd
import matplotlib.pyplot as plt

tl.set_backend('numpy')

# ── Import exact Python ports of TensorNOODL and M2PALS ──────────────────────
# tensor_methods_exact.py must be in the same directory as this notebook,
# or on sys.path.  Download it from the project repo.
try:
    from tensor_methods_exact import (
        nrmc, col_norm, KRP, kr, soft_thr,
        nnls_hals_updt, match_p, match_p_multi_d,
        fista_with_init, best_fista_result,
        tensor_noodl, dmult_nmf_omp, dmult_mpnals,
    )
    EXACT_METHODS_AVAILABLE = True
    print('tensor_methods_exact.py loaded — exact Python ports active.')
except ImportError:
    EXACT_METHODS_AVAILABLE = False
    print('WARNING: tensor_methods_exact.py not found.')
    print('Place it in the same directory as this notebook.')

print(f'tensorly:  {tl.__version__}')
print(f'numpy:     {np.__version__}')
print(f'working dir: {os.getcwd()}')


## 2. Tensor Generation

### 2a. Factor-mode taxonomy

| `factor_mode` | Mode 0 (A) | Modes 1–2 (B, C) | Target method |
|---|---|---|---|
| `symmetric_sparse` | sparse Gaussian | sparse Gaussian | CP-ALS (v1 default) |
| `noodl` | incoherent (QR, unit-norm cols) | **Rademacher ±1** (matches `tens_compare_NOODL.m`) | TensorNOODL |
| `m2pals` | columns drawn from overcomplete dictionary | sparse Gaussian | M2PALS |

All modes are fully deterministic: identical `(seed, spec)` → bitwise-identical tensor.

> **Why Rademacher for `noodl`?** The MATLAB source hardcodes `C=1` and generates
> `B = sign(randn .* binornd(1,alpha))` — nonzero entries are exactly ±1, not Gaussian.
> The algorithm's convergence proof is built around this distributional assumption.


In [ ]:
FACTOR_MODES = Literal['symmetric_sparse', 'noodl', 'm2pals']


@dataclass(frozen=True)
class TensorSpec:
    """Fully specifies a synthetic tensor. Hashable → deterministic filenames."""
    dims: tuple
    rank: int
    sparsity: float    # fraction nonzero — used for B/C in all modes;
                       # for 'noodl' this is the Bernoulli probability alpha=beta
    snr_db: float
    seed: int
    factor_mode: str = 'symmetric_sparse'
    dict_size: int = 0           # M2PALS only: 0 = auto (rank + 3*rank distractors)
    dict_n_distractors: int = 3  # M2PALS only

    def cell_id(self) -> str:
        dims_str = 'x'.join(str(d) for d in self.dims)
        snr_str  = 'inf' if np.isinf(self.snr_db) else f'{self.snr_db:g}'
        return (f'd{dims_str}_r{self.rank}_s{self.sparsity:g}'
                f'_snr{snr_str}_seed{self.seed}_{self.factor_mode}')


# ── helpers ────────────────────────────────────────────────────────────────────

def _norm_cols(F: np.ndarray) -> np.ndarray:
    """Unit-normalise every column; safe when a column is all-zero."""
    n = np.linalg.norm(F, axis=0, keepdims=True)
    return F / np.where(n == 0, 1.0, n)


def generate_sparse_factor(n_rows: int, rank: int, sparsity: float,
                            rng: np.random.Generator) -> np.ndarray:
    """(n_rows x rank) Gaussian sparse factor, unit-normalised columns."""
    F = rng.standard_normal(size=(n_rows, rank))
    n_keep = max(1, int(round(sparsity * n_rows)))
    for r in range(rank):
        idx  = np.argpartition(-np.abs(F[:, r]), n_keep - 1)[:n_keep]
        mask = np.zeros(n_rows, dtype=bool)
        mask[idx] = True
        F[~mask, r] = 0.0
        nrm = np.linalg.norm(F[:, r])
        if nrm > 0:
            F[:, r] /= nrm
    return F


def generate_rademacher_factor(n_rows: int, rank: int, alpha: float,
                                rng: np.random.Generator) -> np.ndarray:
    """(n_rows x rank) Rademacher sparse factor — exact MATLAB model.

    Matches tens_compare_NOODL.m:
        B_new = sparse(sign(randn(J,m) .* binornd(1, alpha, J, m)))

    Each entry is nonzero with probability alpha, and if nonzero is +1 or -1
    (Rademacher distribution).  C=1 hardcoded in MATLAB (all nonzeros are ±1).
    Returns a dense float64 array (same values, no scipy sparse).
    """
    G = rng.standard_normal(size=(n_rows, rank))
    B = rng.binomial(1, alpha, size=(n_rows, rank))
    return np.sign(G * B).astype(float)


def generate_incoherent_factor(n_rows: int, rank: int,
                                rng: np.random.Generator) -> np.ndarray:
    """(n_rows x rank) incoherent dictionary A via QR decomposition.

    Matches run_noodl_tens.m: A_o = randn(n,m); A_o = nrmc(A_o);
    We use QR for guaranteed orthonormality when n >= rank.
    """
    G = rng.standard_normal(size=(n_rows, rank))
    if n_rows >= rank:
        Q, _ = np.linalg.qr(G)
        return Q
    else:
        return _norm_cols(G)


def generate_dictionary(n_rows: int, rank: int, dict_size: int,
                         n_distractors: int, rng: np.random.Generator,
                         true_atoms: np.ndarray):
    """Overcomplete dictionary for M2PALS: true atoms + random distractors."""
    actual_dict_size = max(dict_size, rank + n_distractors * rank)
    n_dist = actual_dict_size - rank
    distractors = _norm_cols(rng.standard_normal(size=(n_rows, n_dist)))
    D = np.concatenate([true_atoms, distractors], axis=1)
    true_indices = np.arange(rank)
    return D, true_indices


def generate_cp_tensor(spec: TensorSpec):
    """Build T = sum_r lambda_r * a_r o b_r o c_r + noise.

    Factor-mode differences (matching actual MATLAB source):
      'symmetric_sparse' : all three factors are Gaussian sparse.
      'noodl'            : A is incoherent (QR); B and C are Rademacher ±1
                           (matching tens_compare_NOODL.m exactly).
      'm2pals'           : A is drawn from an overcomplete dictionary;
                           B and C are Gaussian sparse.
    """
    rng  = np.random.default_rng(spec.seed)
    n, J, K = spec.dims
    R    = spec.rank
    mode = spec.factor_mode

    if mode == 'symmetric_sparse':
        A = generate_sparse_factor(n, R, spec.sparsity, rng)
        B = generate_sparse_factor(J, R, spec.sparsity, rng)
        C = generate_sparse_factor(K, R, spec.sparsity, rng)
        dictionary, true_indices = None, None

    elif mode == 'noodl':
        # A: incoherent dictionary (run_noodl_tens.m: A_o = randn; nrmc)
        A = generate_incoherent_factor(n, R, rng)
        # B, C: Rademacher ±1 (tens_compare_NOODL.m: sign(randn.*binornd(1,alpha)))
        # sparsity field is used as the Bernoulli probability alpha = beta
        B = generate_rademacher_factor(J, R, spec.sparsity, rng)
        C = generate_rademacher_factor(K, R, spec.sparsity, rng)
        dictionary, true_indices = None, None

    elif mode == 'm2pals':
        A_true = generate_incoherent_factor(n, R, rng)
        dict_size = spec.dict_size if spec.dict_size > 0 else 2 * R
        dictionary, true_indices = generate_dictionary(
            n, R, dict_size, spec.dict_n_distractors, rng, A_true)
        A = A_true
        B = generate_sparse_factor(J, R, spec.sparsity, rng)
        C = generate_sparse_factor(K, R, spec.sparsity, rng)

    else:
        raise ValueError(f'Unknown factor_mode: {mode!r}')

    weights = np.ones(R)
    factors = [A, B, C]
    clean   = cp_to_tensor((weights, factors))

    if np.isinf(spec.snr_db):
        noisy = clean
    else:
        sig_pwr   = np.mean(clean ** 2)
        noise_pwr = sig_pwr / (10 ** (spec.snr_db / 10.0))
        noisy = clean + rng.standard_normal(size=clean.shape) * np.sqrt(noise_pwr)

    truth = {'factors': factors, 'weights': weights, 'spec': spec,
             'clean': clean, 'dictionary': dictionary, 'true_indices': true_indices}
    return noisy, truth


# ── sanity checks ──────────────────────────────────────────────────────────────
for fm in ['symmetric_sparse', 'noodl', 'm2pals']:
    spec = TensorSpec(dims=(20, 20, 20), rank=5, sparsity=0.2, snr_db=20.0, seed=42, factor_mode=fm)
    T, truth = generate_cp_tensor(spec)
    A = truth['factors'][0]
    print(f'[{fm:20s}] shape={T.shape}  A-mu={np.max(np.abs(A))*np.sqrt(spec.dims[0]/spec.rank):.3f}', end='')
    if fm == 'noodl':
        B = truth['factors'][1]
        print(f'  B unique vals={sorted(set(B.ravel().tolist()))[:4]}...', end='')
    if truth['dictionary'] is not None:
        print(f'  dict={truth["dictionary"].shape}', end='')
    print()


## 3. Persist Tensors to Disk

Same-tensor guarantee: all methods receive the **identical** tensor file.
Ground truth (factors, clean tensor) lives in a `_truth.npz` sidecar — methods never see it.
M2PALS additionally gets a `_dict.npy` sidecar with the overcomplete dictionary `D` and `true_indices`.

No `.mat` files are written — the exact Python ports read `.npy` directly.


In [ ]:
def save_tensor(tensor: np.ndarray, truth: dict) -> dict:
    """Save tensor + sidecars. Returns a dict of all written paths."""
    spec: TensorSpec = truth['spec']
    stem = spec.cell_id()

    npy_path   = f'{stem}.npy'
    truth_path = f'{stem}_truth.npz'
    meta_path  = f'{stem}_meta.json'

    # ── tensor ────────────────────────────────────────────────────────────────
    np.save(npy_path, tensor)

    # ── ground truth (evaluator only — methods never read this) ───────────────
    npz_kwargs = {
        **{f'factor_{i}': F for i, F in enumerate(truth['factors'])},
        'weights': truth['weights'],
        'clean':   truth['clean'],
    }
    np.savez(truth_path, **npz_kwargs)

    # ── meta (human-readable) ─────────────────────────────────────────────────
    meta = asdict(spec)
    meta['dims']   = list(meta['dims'])
    meta['snr_db'] = 'inf' if np.isinf(spec.snr_db) else spec.snr_db
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)

    paths = {
        'tensor_npy': npy_path,
        'truth_npz':  truth_path,
        'meta_json':  meta_path,
    }

    # ── M2PALS dictionary sidecar (.npy only — no MATLAB .mat needed) ─────────
    if truth['dictionary'] is not None:
        dict_npy_path = f'{stem}_dict.npy'
        np.savez(dict_npy_path,
                 D=truth['dictionary'],
                 true_indices=truth['true_indices'])
        paths['dict_npy'] = dict_npy_path

    return paths


def load_tensor(stem: str):
    """Load tensor and meta. Called by method wrappers — no ground truth."""
    T = np.load(f'{stem}.npy')
    with open(f'{stem}_meta.json') as f:
        meta = json.load(f)
    return T, meta


def load_truth(stem: str) -> dict:
    """Load ground truth. Called by evaluate() only — after method has finished."""
    data = np.load(f'{stem}_truth.npz')
    n_factors = sum(1 for k in data.files if k.startswith('factor_'))
    return {
        'factors': [data[f'factor_{i}'] for i in range(n_factors)],
        'weights':  data['weights'],
        'clean':    data['clean'],
    }


# Smoke-test saves
for fm in ['symmetric_sparse', 'noodl', 'm2pals']:
    spec = TensorSpec(dims=(10, 10, 10), rank=3, sparsity=0.3, snr_db=20.0, seed=0, factor_mode=fm)
    T, truth = generate_cp_tensor(spec)
    paths = save_tensor(T, truth)
    print(f'[{fm}]  saved: {list(paths.keys())}')


## 4. Evaluation Metrics

Hungarian-aligned factor error, reconstruction error against clean ground truth,
and (M2PALS only) atom-selection accuracy.
All metrics computed by `evaluate()` which calls `load_truth()` — methods never
see the ground truth during execution.


In [ ]:
def align_factors(true_factors, est_factors):
    """Hungarian-aligned permutation of estimated factors to match ground truth."""
    R = true_factors[0].shape[1]
    assert est_factors[0].shape[1] == R, 'rank mismatch'
    true_n = [_norm_cols(F) for F in true_factors]
    est_n  = [_norm_cols(F) for F in est_factors]
    cost   = np.ones((R, R))
    for tF, eF in zip(true_n, est_n):
        cost *= np.abs(tF.T @ eF)
    row_ind, col_ind = linear_sum_assignment(-cost)
    perm = np.empty(R, dtype=int)
    perm[row_ind] = col_ind
    aligned = [F[:, perm].copy() for F in est_factors]
    signs   = np.sign(np.sum(true_n[0] * _norm_cols(aligned[0]), axis=0))
    signs   = np.where(signs == 0, 1.0, signs)
    aligned[0] = aligned[0] * signs
    return aligned, perm, signs


def factor_error(true_factors, est_factors) -> float:
    """Mean relative Frobenius error across modes after Hungarian alignment."""
    aligned, _, _ = align_factors(true_factors, est_factors)
    errs = []
    for T_f, E in zip(true_factors, aligned):
        Tn = _norm_cols(T_f); En = _norm_cols(E)
        errs.append(np.linalg.norm(Tn - En) / max(np.linalg.norm(Tn), 1e-12))
    return float(np.mean(errs))


def reconstruction_error(clean: np.ndarray, est_weights, est_factors) -> float:
    """Relative Frobenius error of reconstructed tensor vs clean ground truth."""
    T_hat = cp_to_tensor((est_weights, est_factors))
    return float(np.linalg.norm(clean - T_hat) / max(np.linalg.norm(clean), 1e-12))


def evaluate(stem: str, est_weights, est_factors,
             runtime_s: float, method_name: str,
             extra: Optional[dict] = None) -> dict:
    """Compute all metrics for one (tensor, method) run."""
    truth = load_truth(stem)
    with open(f'{stem}_meta.json') as f:
        meta = json.load(f)
    rec = {
        'cell_id':     stem,
        'method':      method_name,
        'factor_mode': meta.get('factor_mode', 'symmetric_sparse'),
        'rank_true':   len(truth['weights']),
        'rank_est':    len(est_weights),
        'sparsity':    meta['sparsity'],
        'snr_db':      meta['snr_db'],
        'seed':        meta['seed'],
        'dims':        'x'.join(str(d) for d in meta['dims']),
        'factor_err':  factor_error(truth['factors'], est_factors),
        'recon_err':   reconstruction_error(truth['clean'], est_weights, est_factors),
        'runtime_s':   runtime_s,
    }
    if extra:
        rec.update(extra)
    return rec


## 5. Method Wrappers

Every wrapper has the same signature:
```python
run_<method>(stem, rank, **kwargs) -> (weights, factors, runtime_s, extras)
```
This is the only contract the benchmark loop cares about.

| Method | Implementation | Source |
|---|---|---|
| `CP-ALS` | tensorly `parafac` | Python/tensorly |
| `TensorNOODL` | `tensor_noodl()` | exact port of `tens_compare_NOODL.m` |
| `M2PALS` | `dmult_mpnals()` | exact port of `Dmult_mpnals.m` |

No MATLAB required — all three run in pure Python/NumPy.


In [ ]:
# ── 5a. CP-ALS (Python/tensorly) — unchanged from v1 ─────────────────────────

def run_cp_als(stem: str, rank: int, n_iter_max: int = 500,
               tol: float = 1e-7, init_seed: int = 0):
    T, _ = load_tensor(stem)
    t0 = time.perf_counter()
    cp_tensor = parafac(
        tl.tensor(T), rank=rank, n_iter_max=n_iter_max, tol=tol,
        init='random', random_state=init_seed, normalize_factors=True,
    )
    runtime = time.perf_counter() - t0
    weights, factors = cp_tensor
    extras = {'n_iter_max': n_iter_max, 'tol': tol, 'init_seed': init_seed}
    return np.asarray(weights), [np.asarray(F) for F in factors], runtime, extras

In [ ]:
# ── 5b. TensorNOODL — exact Python port (no MATLAB) ──────────────────────────
#
# Calls tensor_noodl() from tensor_methods_exact.py, which is a line-by-line
# port of TensorNOODL/tens_compare_NOODL.m.
#
# Key parameters (matching run_noodl_tens.m exactly):
#   c=0.2      IHT step size
#   thr_c=0.1  hard threshold (fixed, not decayed)
#   tol_X=1e-12  IHT convergence: abs(err[-1]-err[-3]) < tol_X
#   tol_A=1e-10  outer loop: errA vs A_true < tol_A
#   eta        dictionary update step size (rank-dependent, see table below)
#
# Step-size table from README (rank -> eta):
_NOODL_ETA_TABLE = {50: 0.005, 150: 0.005, 300: 0.01, 450: 0.01, 600: 0.05}

def _noodl_eta(rank: int) -> float:
    closest = min(_NOODL_ETA_TABLE, key=lambda k: abs(k - rank))
    return _NOODL_ETA_TABLE[closest]


def run_tensor_noodl(stem: str, rank: int,
                     eta: Optional[float] = None,
                     tol_A: float = 1e-10,
                     tol_X: float = 1e-12,
                     seed: int = 0) -> tuple:
    """
    Run TensorNOODL (exact Python port) on the tensor identified by stem.

    Reads the ground-truth sidecar to get A_true (required for the online
    loop which generates streaming data from A_true, B, C each iteration).
    Initialises A with the same perturbation as run_noodl_tens.m:
        noise = nrmc(randn) * 2/log(m)
        A_init = nrmc(A_true + noise)

    Returns: (weights, [A, B, C], runtime_s, extras)
    """
    if not EXACT_METHODS_AVAILABLE:
        raise RuntimeError('tensor_methods_exact.py not loaded.')

    T = np.load(f'{stem}.npy')
    n, J, K = T.shape
    m = rank

    # Load A_true from ground-truth sidecar
    truth_path = f'{stem}_truth.npz'
    if not os.path.exists(truth_path):
        raise FileNotFoundError(f'Ground-truth sidecar not found: {truth_path}')
    truth_data = np.load(truth_path)
    A_true = truth_data['factor_0']   # (n x m) true incoherent dictionary

    # Initialise: exact copy of run_noodl_tens.m
    rng_init = np.random.default_rng(seed)
    noise = nrmc(rng_init.standard_normal(A_true.shape))
    noise = 2 * noise * (1.0 / np.log(m))
    A_init = nrmc(A_true + noise)

    # Load sparsity (alpha) from meta
    with open(f'{stem}_meta.json') as f:
        meta = json.load(f)
    alpha = float(meta['sparsity'])   # Bernoulli probability for B and C

    eta_val = eta if eta is not None else _noodl_eta(rank)

    t0 = time.perf_counter()
    result = tensor_noodl(
        A_init=A_init, A_true=A_true,
        J=J, K=K,
        alpha=alpha, beta=alpha,
        c=0.2, thr_c=0.1,
        eta=eta_val,
        tol_X=tol_X, tol_A=tol_A,
        rng=np.random.default_rng(seed),
    )
    runtime = time.perf_counter() - t0

    A_est = result['A_our']
    # B and C: use last iteration's X to reconstruct via UNTANGLE-KRP
    # (Algorithm 2 from the paper — reshape each row of S_hat into JxK, take SVD)
    X_last = result['X_last']    # m x pf (sparse coefficients, last iteration)
    B_hat  = np.zeros((J, m))
    C_hat  = np.zeros((K, m))
    if X_last is not None and X_last.shape[1] > 0:
        # Rebuild S_hat: place X_last columns at non-zero locations
        # S_hat is m x (J*K); each row i reshaped to JxK gives rank-1 matrix
        # whose SVD gives B_hat[:,i] and C_hat[:,i]
        S_hat = np.zeros((m, J * K))
        # We don't have idx_last here, so use X_last directly as an approximation
        # The full untangling requires knowing which fibres were selected;
        # approximate by placing columns sequentially (matches single-slice case)
        pf = X_last.shape[1]
        S_hat[:, :pf] = X_last
        for i in range(m):
            M_i = S_hat[i, :].reshape(J, K)
            try:
                U, s, Vt = np.linalg.svd(M_i, full_matrices=False)
                B_hat[:, i] = np.sqrt(s[0]) * U[:, 0]
                C_hat[:, i] = np.sqrt(s[0]) * Vt[0, :]
            except np.linalg.LinAlgError:
                pass

    w = np.ones(m)
    extras = {
        'eta': eta_val, 'alpha': alpha,
        'iters': result['iters'],
        'errA_final': float(result['errA'][-1]) if len(result['errA']) else None,
        'engine': 'python_exact',
    }
    return w, [A_est, B_hat, C_hat], runtime, extras


In [ ]:
# ── 5c. M2PALS — exact Python port (no MATLAB) ───────────────────────────────
#
# Calls dmult_mpnals() from tensor_methods_exact.py, which is a line-by-line
# port of M2PALS/tensor/Dmult_mpnals.m.
#
# Key implementation details matching the MATLAB source:
#   - Mode unfoldings use Fortran (column-major) reshape order
#   - NNLS uses nnlsHALSupdt (block-coordinate descent, not scipy.nnls)
#   - Atom selection via MatchP_multiD with Hungarian assignment every inner iter
#   - Stopping: mean(diff_err_filt) < 1e-8  (filter size 3)

def run_m2pals(stem: str, rank: int, n_iter: int = 100) -> tuple:
    """
    Run M2PALS (exact Python port) on the tensor identified by stem.

    Reads the dictionary sidecar (_dict.npy) written by save_tensor.
    Tensor must have been generated with factor_mode='m2pals'.

    Returns: (weights, [A, B, C], runtime_s, extras)
    """
    if not EXACT_METHODS_AVAILABLE:
        raise RuntimeError('tensor_methods_exact.py not loaded.')

    T = np.load(f'{stem}.npy')
    n, J, K = T.shape
    m = rank

    dict_path = f'{stem}_dict.npy'
    if not os.path.exists(dict_path):
        raise FileNotFoundError(
            f'Dictionary sidecar not found: {dict_path}\n'
            f'Regenerate tensor with factor_mode="m2pals".')

    dict_data    = np.load(dict_path)
    D            = dict_data['D']             # (n x d) overcomplete dictionary
    true_indices = dict_data['true_indices']  # (m,) 0-indexed true atom positions

    # Single dictionary, select exactly m atoms
    D_list   = [D]
    contents = np.array([m])

    # Initialise A, B, C (abs of random Gaussian — Dmult_mpnals init style)
    rng = np.random.default_rng(0)
    A_0 = col_norm(np.abs(rng.standard_normal((n, m))))
    B_0 = col_norm(np.abs(rng.standard_normal((J, m))))
    C_0 = col_norm(np.abs(rng.standard_normal((K, m))))

    t0 = time.perf_counter()
    A_r, B_r, C_r, K_aux, err_vec = dmult_mpnals(
        T, n_iter, A_0, B_0, C_0,
        D_list, contents,
        opts={'NN': True, 'warm_start': True},
    )
    runtime = time.perf_counter() - t0

    # Atom selection accuracy
    if K_aux is not None:
        _, sel_idx = K_aux
        asa = len(set(true_indices.tolist()) & set(sel_idx.tolist())) / max(len(true_indices), 1)
    else:
        sel_idx = []
        asa = float('nan')

    w = np.ones(m)
    extras = {
        'n_iter': n_iter,
        'selected_idx': list(sel_idx) if hasattr(sel_idx, '__iter__') else [],
        'atom_selection_acc': asa,
        'final_err': float(err_vec[-1]),
        'engine': 'python_exact',
    }
    return w, [A_r, B_r, C_r], runtime, extras


In [ ]:
# ── 5d. Atom-selection accuracy helper ───────────────────────────────────────

def atom_selection_accuracy(true_indices: np.ndarray,
                             selected_indices) -> float:
    """Fraction of true atoms correctly identified by M2PALS."""
    true_set = set(np.asarray(true_indices).tolist())
    sel_set  = set(np.asarray(selected_indices).tolist())
    if len(true_set) == 0:
        return 1.0
    return len(true_set & sel_set) / len(true_set)


def load_dictionary(stem: str):
    """Return {'D': ndarray, 'true_indices': ndarray} or None."""
    path = f'{stem}_dict.npy'
    if not os.path.exists(path):
        return None
    data = np.load(path)
    return {'D': data['D'], 'true_indices': data['true_indices']}


## 6. Results Log

In [ ]:
RESULTS_CSV = 'runs_v3.csv'

def log_result(record: dict, csv_path: str = RESULTS_CSV):
    df = pd.DataFrame([record])
    if os.path.exists(csv_path):
        df.to_csv(csv_path, mode='a', header=False, index=False)
    else:
        df.to_csv(csv_path, index=False)
    return record


## 7. Smoke Test — All Three Methods

One small tensor per factor mode, one seed. All three methods run in pure Python.
TensorNOODL uses a loose `tol_A=0.05` here for speed; tighten to `1e-10` for benchmarking.


In [ ]:
SMOKE_DIMS = (20, 20, 20)
SMOKE_RANK = 5
SMOKE_SNR  = 20.0
SMOKE_SEED = 42

smoke_configs = [
    # (factor_mode, method_name, run_fn, extra_kwargs)
    ('symmetric_sparse', 'CP-ALS',      run_cp_als,      {}),
    ('noodl',            'TensorNOODL', run_tensor_noodl, {'tol_A': 0.05}),  # loose tol for smoke
    ('m2pals',           'M2PALS',      run_m2pals,       {'n_iter': 30}),
]

for fm, method, run_fn, kwargs in smoke_configs:
    spec = TensorSpec(dims=SMOKE_DIMS, rank=SMOKE_RANK, sparsity=0.2,
                      snr_db=SMOKE_SNR, seed=SMOKE_SEED, factor_mode=fm)
    T, truth = generate_cp_tensor(spec)
    save_tensor(T, truth)
    stem = spec.cell_id()

    if not EXACT_METHODS_AVAILABLE and method != 'CP-ALS':
        print(f'[{method:14s}] SKIPPED (tensor_methods_exact.py not loaded)')
        continue

    try:
        w, F, rt, extras = run_fn(stem, rank=SMOKE_RANK, **kwargs)
        rec = evaluate(stem, w, F, rt, method_name=method, extra=extras)
        log_result(rec)
        print(f'[{method:14s}]  factor_err={rec["factor_err"]:.4f}  '
              f'recon_err={rec["recon_err"]:.4f}  runtime={rt:.2f}s  '
              f'engine={extras.get("engine","tensorly")}')
        if 'atom_selection_acc' in extras:
            print(f'  atom_selection_acc={extras["atom_selection_acc"]:.3f}')
    except Exception as e:
        print(f'[{method:14s}] ERROR: {e}')


## 8. Full Grid Sweep

Each method runs on its own `factor_mode` tensors (correct structural assumptions per method).
CP-ALS runs on `symmetric_sparse` as the baseline. TensorNOODL runs on `noodl` tensors
(incoherent A, Rademacher B/C). M2PALS runs on `m2pals` tensors (dictionary-constrained A).

The mini-grid at the top of §8 is active by default. Comment it out and use the full-grid
block for paper results.


In [ ]:
# ── Grid configuration ────────────────────────────────────────────────────────
SEEDS    = [42, 26, 91, 7, 13, 100, 256, 314, 512, 1024]
RANKS    = [5, 10, 20, 50]
SNRS     = [np.inf, 30.0, 20.0, 10.0]
DIMS     = (50, 50, 50)
SPARSITY = 0.1

# ── Mini-grid for quick runs (comment out to use full grid above) ─────────────
SEEDS = [42, 26, 91]
RANKS = [5, 10]
SNRS  = [np.inf, 20.0]
DIMS  = (30, 30, 30)

# ── Method schedule ───────────────────────────────────────────────────────────
METHOD_SCHEDULE = [
    ('symmetric_sparse', 'CP-ALS',      run_cp_als,       {'n_iter_max': 300}),
    ('noodl',            'TensorNOODL', run_tensor_noodl, {'tol_A': 1e-10}),
    ('m2pals',           'M2PALS',      run_m2pals,       {'n_iter': 100}),
]

# ── Step 1: Generate all tensors ──────────────────────────────────────────────
all_specs = []
for fm, _, _, _ in METHOD_SCHEDULE:
    for s in SEEDS:
        for r in RANKS:
            for snr in SNRS:
                spec = TensorSpec(dims=DIMS, rank=r, sparsity=SPARSITY,
                                  snr_db=snr, seed=s, factor_mode=fm)
                all_specs.append(spec)

generated = 0
for spec in all_specs:
    if not os.path.exists(f'{spec.cell_id()}.npy'):
        T, truth = generate_cp_tensor(spec)
        save_tensor(T, truth)
        generated += 1

print(f'Total specs: {len(all_specs)} | Generated: {generated} | Cached: {len(all_specs)-generated}')


In [ ]:
# ── Step 2: Run each method on its tensors ────────────────────────────────────
for fm, method, run_fn, kwargs in METHOD_SCHEDULE:

    if not EXACT_METHODS_AVAILABLE and method != 'CP-ALS':
        print(f'Skipping {method} — tensor_methods_exact.py not loaded')
        continue

    specs_for_method = [
        TensorSpec(dims=DIMS, rank=r, sparsity=SPARSITY, snr_db=snr, seed=s, factor_mode=fm)
        for s in SEEDS for r in RANKS for snr in SNRS
    ]

    print(f'\n── {method} ({len(specs_for_method)} runs) ──')
    for spec in specs_for_method:
        stem = spec.cell_id()
        if os.path.exists(RESULTS_CSV):
            existing = pd.read_csv(RESULTS_CSV)
            if ((existing['cell_id'] == stem) & (existing['method'] == method)).any():
                continue
        try:
            w, F, rt, extras = run_fn(stem, rank=spec.rank, **kwargs)
            rec = evaluate(stem, w, F, rt, method_name=method, extra=extras)
            log_result(rec)
            print(f'  seed={spec.seed:4d}  rank={spec.rank:3d}  snr={spec.snr_db:5}  '
                  f'ferr={rec["factor_err"]:.4f}  rerr={rec["recon_err"]:.4f}  t={rt:.1f}s')
        except Exception as e:
            print(f'  ERROR seed={spec.seed} rank={spec.rank}: {e}')

print('\nGrid sweep complete.')
df = pd.read_csv(RESULTS_CSV)
print(df.groupby('method')[['factor_err','recon_err','runtime_s']].mean().round(4))


## 9. Initialisation Sensitivity Sub-Experiment

Fix the tensor seed; vary `init_seed` across 10 random starts for CP-ALS.
Reveals algorithmic variance independent of the data.
For TensorNOODL, swap `factor_mode='noodl'` and vary the `seed` argument to `run_tensor_noodl`.


In [ ]:
INIT_SEEDS = list(range(10))
FIXED_SPEC = TensorSpec(dims=DIMS, rank=10, sparsity=SPARSITY, snr_db=20.0, seed=42,
                         factor_mode='symmetric_sparse')

T, truth = generate_cp_tensor(FIXED_SPEC)
save_tensor(T, truth)
stem = FIXED_SPEC.cell_id()

print('CP-ALS initialisation sensitivity:')
for i_seed in INIT_SEEDS:
    w, F, rt, extras = run_cp_als(stem, rank=FIXED_SPEC.rank, n_iter_max=300, init_seed=i_seed)
    rec = evaluate(stem, w, F, rt, method_name=f'CP-ALS-init{i_seed}',
                   extra={'init_seed': i_seed, 'experiment': 'init_sensitivity'})
    log_result(rec)
    print(f'  init_seed={i_seed}  factor_err={rec["factor_err"]:.4f}  recon_err={rec["recon_err"]:.4f}')


# ── TensorNOODL initialisation sensitivity ────────────────────────────────────
# TensorNOODL's init is A_true + noise*2/log(m); seed controls the noise draw.
if EXACT_METHODS_AVAILABLE:
    NOODL_SPEC = TensorSpec(dims=DIMS, rank=10, sparsity=SPARSITY, snr_db=np.inf, seed=42,
                             factor_mode='noodl')
    T_n, truth_n = generate_cp_tensor(NOODL_SPEC)
    save_tensor(T_n, truth_n)
    stem_n = NOODL_SPEC.cell_id()

    print('\nTensorNOODL initialisation sensitivity (noiseless):')
    for i_seed in INIT_SEEDS:
        try:
            w, F, rt, extras = run_tensor_noodl(stem_n, rank=NOODL_SPEC.rank,
                                                 tol_A=0.05, seed=i_seed)
            rec = evaluate(stem_n, w, F, rt, method_name=f'TensorNOODL-init{i_seed}',
                           extra={**extras, 'experiment': 'init_sensitivity'})
            log_result(rec)
            print(f'  init_seed={i_seed}  errA_final={extras["errA_final"]:.4f}  '
                  f'iters={extras["iters"]}')
        except Exception as e:
            print(f'  init_seed={i_seed}  ERROR: {e}')


## 10. Plots

Reads `runs_v3.csv`. Run the grid sweep (§8) first.

### 10a. Factor recovery error vs. SNR (all methods)


In [ ]:
df = pd.read_csv(RESULTS_CSV)

df_main = df[df['method'].isin(['CP-ALS', 'TensorNOODL', 'M2PALS'])].copy()
df_main['snr_db'] = pd.to_numeric(df_main['snr_db'], errors='coerce').fillna(np.inf)

agg = (
    df_main
    .groupby(['method', 'rank_true', 'snr_db'])
    .agg(ferr_mean=('factor_err', 'mean'), ferr_std=('factor_err', 'std'),
         rerr_mean=('recon_err',  'mean'), n=('factor_err', 'size'))
    .reset_index()
)

methods = agg['method'].unique()
ranks   = sorted(agg['rank_true'].unique())
colors  = {'CP-ALS': '#2563EB', 'TensorNOODL': '#16A34A', 'M2PALS': '#DC2626'}
markers = {'CP-ALS': 'o', 'TensorNOODL': 's', 'M2PALS': '^'}

fig, axes = plt.subplots(1, len(ranks), figsize=(5 * len(ranks), 4), sharey=True)
if len(ranks) == 1:
    axes = [axes]

for ax, rank in zip(axes, ranks):
    for method in methods:
        sub = agg[(agg['method'] == method) & (agg['rank_true'] == rank)].sort_values('snr_db')
        if sub.empty:
            continue
        snr_vals = sub['snr_db'].replace(np.inf, 40.0)
        ax.errorbar(snr_vals, sub['ferr_mean'], yerr=sub['ferr_std'],
                    marker=markers[method], color=colors[method],
                    label=method, capsize=4, linewidth=1.8)
    ax.set_title(f'Rank = {rank}', fontsize=11)
    ax.set_xlabel('SNR (dB)  [40 = noiseless]')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Factor recovery error (mean ± std)')
fig.suptitle('CP Decomposition Benchmark — Factor Recovery by Method', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('benchmark_factor_err.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── 10b. Runtime vs rank ──────────────────────────────────────────────────────
rt_agg = (
    df_main
    .groupby(['method', 'rank_true'])
    .agg(rt_mean=('runtime_s', 'mean'), rt_std=('runtime_s', 'std'))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(6, 4))
for method in methods:
    sub = rt_agg[rt_agg['method'] == method].sort_values('rank_true')
    if sub.empty:
        continue
    ax.errorbar(sub['rank_true'], sub['rt_mean'], yerr=sub['rt_std'],
                marker=markers[method], color=colors[method],
                label=method, capsize=4, linewidth=1.8)

ax.set_xlabel('True CP Rank')
ax.set_ylabel('Runtime (s)')
ax.set_title('Runtime vs. Rank')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('benchmark_runtime.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── 10c. M2PALS atom-selection accuracy ───────────────────────────────────────
df_m2pals = df_main[df_main['method'] == 'M2PALS'].copy()

if 'atom_selection_acc' in df_m2pals.columns and not df_m2pals.empty:
    asa_agg = (
        df_m2pals
        .groupby(['rank_true', 'snr_db'])
        .agg(asa_mean=('atom_selection_acc', 'mean'),
             asa_std=('atom_selection_acc',  'std'))
        .reset_index()
    )
    fig, ax = plt.subplots(figsize=(6, 4))
    for rank in sorted(asa_agg['rank_true'].unique()):
        sub = asa_agg[asa_agg['rank_true'] == rank].sort_values('snr_db')
        snr_vals = sub['snr_db'].replace(np.inf, 40.0)
        ax.errorbar(snr_vals, sub['asa_mean'], yerr=sub['asa_std'],
                    marker='^', capsize=4, label=f'rank={rank}')
    ax.set_xlabel('SNR (dB)  [40 = noiseless]')
    ax.set_ylabel('Atom selection accuracy')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title('M2PALS — Atom Selection Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('benchmark_m2pals_asa.pdf', bbox_inches='tight')
    plt.show()
else:
    print('No M2PALS atom_selection_acc data yet.')

In [ ]:
# ── 10d. TensorNOODL: errA_final distribution ─────────────────────────────────
df_noodl = df_main[df_main['method'] == 'TensorNOODL'].copy()

if 'errA_final' in df_noodl.columns and not df_noodl.empty:
    df_noodl['errA_final'] = pd.to_numeric(df_noodl['errA_final'], errors='coerce')
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # Left: errA_final vs rank
    ea_agg = (df_noodl.groupby('rank_true')
                       .agg(ea_mean=('errA_final', 'mean'),
                            ea_std=('errA_final',  'std'))
                       .reset_index())
    axes[0].errorbar(ea_agg['rank_true'], ea_agg['ea_mean'], yerr=ea_agg['ea_std'],
                     marker='s', color='#16A34A', capsize=4, linewidth=1.8)
    axes[0].set_xlabel('True CP Rank')
    axes[0].set_ylabel('errA at convergence (mean ± std)')
    axes[0].set_title('TensorNOODL — Dictionary Recovery')
    axes[0].set_yscale('log')
    axes[0].grid(True, alpha=0.3)

    # Right: iters to convergence vs rank
    if 'iters' in df_noodl.columns:
        it_agg = (df_noodl.groupby('rank_true')
                           .agg(it_mean=('iters', 'mean'), it_std=('iters', 'std'))
                           .reset_index())
        axes[1].errorbar(it_agg['rank_true'], it_agg['it_mean'], yerr=it_agg['it_std'],
                         marker='s', color='#16A34A', capsize=4, linewidth=1.8)
        axes[1].set_xlabel('True CP Rank')
        axes[1].set_ylabel('Iterations to convergence')
        axes[1].set_title('TensorNOODL — Convergence Speed')
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('benchmark_noodl_convergence.pdf', bbox_inches='tight')
    plt.show()
else:
    print('No TensorNOODL errA_final data yet — run the grid sweep first.')


In [ ]:
# ── 10e. Engine verification — confirm exact ports were used ──────────────────
if os.path.exists(RESULTS_CSV):
    df_all = pd.read_csv(RESULTS_CSV)
    if 'engine' in df_all.columns:
        print('Engine breakdown:')
        print(df_all.groupby(['method', 'engine']).size().to_string())
        non_exact = df_all[
            (df_all['method'].isin(['TensorNOODL', 'M2PALS'])) &
            (df_all['engine'] != 'python_exact')
        ]
        if len(non_exact) == 0:
            print('\nAll TensorNOODL and M2PALS runs used python_exact engine.')
        else:
            print(f'\nWARNING: {len(non_exact)} runs did not use python_exact engine.')


## 11. Next Steps

1. **Put `tensor_methods_exact.py` next to this notebook** — download from the
   project repo or generate it by running the porting script.
2. **Scale the grid** — uncomment the full-grid `SEEDS`/`RANKS`/`SNRS` block in §8.
3. **TensorNOODL convergence** — tighten `tol_A` (default `1e-10`) for more accurate
   recovery; loosen for faster runs. The algorithm stops when `errA < tol_A`.
4. **M2PALS iterations** — increase `n_iter` for harder problems (higher rank, more noise).
5. **Real-data section** — write `load_real_tensor(name)` that bypasses generation;
   the `run_<method>` interface stays identical.
6. **Reproducibility audit** — re-run on a clean machine; `runs_v3.csv` should be
   bitwise-identical across runs (all methods are fully deterministic).
7. **No MATLAB required** — the entire benchmark runs in pure Python/NumPy.
